In [1]:
import pandas as pd
import numpy as np

application_bureau_prev = pd.read_csv(
    "../data/processed/application_bureau_prev.csv"
)

inst = pd.read_csv(
    "../data/raw/installments_payments.csv"
)

In [2]:
inst["DAYS_LATE"] = (
    inst["DAYS_ENTRY_PAYMENT"]
    - inst["DAYS_INSTALMENT"]
)

inst["DAYS_LATE"] = (
    inst["DAYS_LATE"]
    .clip(lower=0)
)

inst["PAYMENT_RATIO"] = (
    inst["AMT_PAYMENT"]
    /
    inst["AMT_INSTALMENT"]
)

inst["PAYMENT_RATIO"] = (
    inst["PAYMENT_RATIO"]
    .replace([np.inf,-np.inf],np.nan)
)

inst["PAYMENT_DIFF"] = (
    inst["AMT_PAYMENT"]
    - inst["AMT_INSTALMENT"]
)

inst["LATE_PAYMENT"] = (
    inst["DAYS_LATE"] > 0
).astype(int)

inst["LATE_30"] = (
    inst["DAYS_LATE"] > 30
).astype(int)

inst["LATE_90"] = (
    inst["DAYS_LATE"] > 90
).astype(int)

In [3]:
inst_features = inst.groupby(
    "SK_ID_CURR"
).agg(

    inst_num_loans=(
        "SK_ID_PREV",
        "nunique"
    ),

    inst_avg_days_late=(
        "DAYS_LATE",
        "mean"
    ),

    inst_max_days_late=(
        "DAYS_LATE",
        "max"
    ),

    inst_total_days_late=(
        "DAYS_LATE",
        "sum"
    ),

    inst_late_payment_ratio=(
        "LATE_PAYMENT",
        "mean"
    ),

    inst_late30_ratio=(
        "LATE_30",
        "mean"
    ),

    inst_late90_ratio=(
        "LATE_90",
        "mean"
    ),

    inst_avg_payment_ratio=(
        "PAYMENT_RATIO",
        "mean"
    ),

    inst_min_payment_ratio=(
        "PAYMENT_RATIO",
        "min"
    ),

    inst_max_payment_ratio=(
        "PAYMENT_RATIO",
        "max"
    ),

    inst_avg_payment=(
        "AMT_PAYMENT",
        "mean"
    ),

    inst_total_payment=(
        "AMT_PAYMENT",
        "sum"
    ),

    inst_avg_installment=(
        "AMT_INSTALMENT",
        "mean"
    ),

    inst_total_installment=(
        "AMT_INSTALMENT",
        "sum"
    ),

    inst_avg_payment_diff=(
        "PAYMENT_DIFF",
        "mean"
    ),

    inst_total_payment_diff=(
        "PAYMENT_DIFF",
        "sum"
    )

).reset_index()

In [4]:
inst_features = inst_features.replace(
    [np.inf,-np.inf],
    np.nan
)

In [5]:
application_bureau_prev_inst = (
    application_bureau_prev.merge(
        inst_features,
        on="SK_ID_CURR",
        how="left"
    )
)

In [7]:
application_bureau_prev_inst.to_csv(
    "../data/processed/application_bureau_prev_inst.csv",
    index=False
)

In [8]:
print(
    application_bureau_prev_inst.shape
)

print(
    [c for c in application_bureau_prev_inst.columns
     if c.startswith("inst_")]
)

(307511, 157)
['inst_num_loans', 'inst_avg_days_late', 'inst_max_days_late', 'inst_total_days_late', 'inst_late_payment_ratio', 'inst_late30_ratio', 'inst_late90_ratio', 'inst_avg_payment_ratio', 'inst_min_payment_ratio', 'inst_max_payment_ratio', 'inst_avg_payment', 'inst_total_payment', 'inst_avg_installment', 'inst_total_installment', 'inst_avg_payment_diff', 'inst_total_payment_diff']
